**Generate the Name-Search Query Sets**

This notebook completes the preparation stage of the experiment design (`./docs/experiment_design.pdf`, §2.1 and Table 1) by generating the five query sets of the manifest (`./docs/MANIFEST.md`, §4.5) from the target persons that `step_2_prepare_evaluation_set.ipynb` wrote. It reads `./eval/persons_dev.csv`, `./eval/persons_test.csv` and `./results/preflight.json`, and never modifies them or the transcription. Every random step is seeded, so a re-run writes identical files. A development query and its relevant persons involve development persons only, so that the development notebooks never meet a test identifier. It outputs:

1. `./eval/queries_dev.csv` and `./eval/queries_test.csv`: one row per query, with its set, text, script, target and relevant persons, masked and source fields, split and fold, and its tokens as JSON
2. `./review/qs2_sample.csv`: 50 random diaspora spellings beside the recorded names, left for human review

**1. Configuration and Helpers**

This cell defines the settings and helpers that every later cell reuses, and reads the target persons.

1. `SEED`: the seed of every random step (typing errors, spelling variants and sampling), 333 as the manifest requires
2. `normalise_latin`, `normalise_hebrew` and `token_script`: copies of the normalisation of `step_2_prepare_evaluation_set.ipynb`, which a query token needs once a typing error or a variant has changed its spelling. The consistency check confirms that the copies reproduce every stored token's normalised form
3. `persons`: one `Person` per target person of both splits, with its name tokens as `NameToken`s

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# A query is one CSV row. Its tokens are stored as JSON, each with:
#   • display      the text typed (a typing error or variant stays in it)
#   • norm/script  its normalised form and script, as the index would see them
#   • field        given or surname: the field of the recorded token
#   • source_norm  the normalised recorded token the query token stands for
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
EVAL_DIR        = Path("eval")                       # persons in, queries out
REVIEW_DIR      = Path("review")                     # human-review files
PREFLIGHT_FILE  = Path("results/preflight.json")     # asset statuses
SEED            = 333          # every random step
QS5_SIZE        = 300          # partial-name queries sampled from QS1–QS4
QS5_SURNAME_P   = 0.6          # chance that a QS5 query keeps the surname
QS2_REVIEW_SIZE = 50           # QS2 variants written for human review
QS2_LLM_MODEL   = "claude-opus-5-5"   # QS2 fallback, only when ANTHROPIC_API_KEY is set
RULE            = "─" * 72     # frames every printed report

FINAL_TO_MEDIAL   = str.maketrans("ךםןףץ", "כמנפצ")     # final letters → medial forms
MAQAF             = "\N{HEBREW PUNCTUATION MAQAF}"     # the Hebrew hyphen
HEBREW_LETTERS    = "\N{HEBREW LETTER ALEF}-\N{HEBREW LETTER TAV}"            # a regex range
HEBREW_MARKS      = re.compile("[\N{HEBREW ACCENT ETNAHTA}-\N{HEBREW POINT QAMATS QATAN}]")
HEBREW_NON_LETTER = re.compile(f"[^{HEBREW_LETTERS} ]")  # anything but a letter or space
YIDDISH_LIGATURES = {"\N{HEBREW LIGATURE YIDDISH DOUBLE VAV}": "וו",
                     "\N{HEBREW LIGATURE YIDDISH VAV YOD}": "וי",
                     "\N{HEBREW LIGATURE YIDDISH DOUBLE YOD}": "יי"}


@dataclass(frozen=True)
class NameToken:
    """One token of a person's name: its `field` (given or surname), source
    `column`, `script` (latin or hebrew), normalised form `norm` and recorded
    `display` spelling, as `step_2_prepare_evaluation_set.ipynb` stored them."""

    field: str
    column: str
    script: str
    norm: str
    display: str


@dataclass(frozen=True)
class Person:
    """One target person: its id, section, split, fold (0 for test) and tokens."""

    person_id: str
    section: str
    split: str
    fold: int
    tokens: tuple[NameToken, ...]


def token_script(text: str) -> str:
    """Return `hebrew` if `text` holds any Hebrew character, else `latin`."""
    return "hebrew" if any(0x0590 <= ord(char) <= 0x05FF for char in text) else "latin"


def normalise_latin(text: str) -> str:
    """Normalise Latin-script text: no accents, lower case, letters only."""
    decomposed = unicodedata.normalize("NFKD", text.replace("-", " "))
    letters = "".join(char for char in decomposed
                      if not unicodedata.combining(char) and (char.isalpha() or char.isspace()))
    return " ".join(letters.lower().split())


def normalise_hebrew(text: str) -> str:
    """Normalise Hebrew-script text: no points, medial letters, letters only."""
    text = unicodedata.normalize("NFKD", text).replace(MAQAF, " ").replace("-", " ")
    text = HEBREW_MARKS.sub("", text)
    for ligature, letters in YIDDISH_LIGATURES.items():
        text = text.replace(ligature, letters)
    text = HEBREW_NON_LETTER.sub("", text.translate(FINAL_TO_MEDIAL))
    return " ".join(text.split())


def normalise(text: str) -> str:
    """Normalise one token in its own script, without spaces."""
    script_normaliser = normalise_hebrew if token_script(text) == "hebrew" else normalise_latin
    return script_normaliser(text).replace(" ", "")


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def read_persons(split: str) -> list[Person]:
    """Read the target persons of `split` (`dev` or `test`)."""
    return [Person(row["person_id"], row["section"], row["split"], int(row["fold"] or 0),
                   tuple(NameToken(**token) for token in json.loads(row["tokens"])))
            for row in read_csv(EVAL_DIR / f"persons_{split}.csv")]


preflight = json.loads(PREFLIGHT_FILE.read_text(encoding="utf-8"))
dev_persons, test_persons = read_persons("dev"), read_persons("test")
persons = sorted(dev_persons + test_persons, key=lambda person: person.person_id)
persons_by_id = {person.person_id: person for person in persons}
print(f"✓ Configuration and helpers loaded ({len(dev_persons)} development and "
      f"{len(test_persons)} test persons, seed {SEED}).")

✓ Configuration and helpers loaded (507 development and 130 test persons, seed 333).


**2. Query Sets QS1–QS4**

This cell takes `persons` and generates one query per eligible person in each of the first four query sets of the design's Table 1:

1. **QS1, Latin name with typing errors**: the recorded Latin given name and surname, in either order with equal chance, with one token edited once (five letters or fewer) or twice (longer). Each edit is a deletion, a swap of adjacent letters or a substitution by a neighbouring key on a QWERTY keyboard
2. **QS2, diaspora spelling**: the same name rewritten by the substitution rules of the manifest's Appendix C, one applicable rule per token chosen at random (`Kofman` → `Koffman`). A person with no applicable rule is sent to the LLM fallback when `ANTHROPIC_API_KEY` is set, and is otherwise dropped and counted
3. **QS3, Hebrew → Latin record**: the recorded Hebrew given name and surname, with the target's Hebrew tokens masked, so that only its Latin record can match
4. **QS4, Latin → Hebrew record**: the mirror image of QS3

It produces `queries` and `query_drops`, the persons dropped per set.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Query sets QS1–QS4  —  typing errors, diaspora spellings, cross-script
#
# Masking removes the target's tokens of the listed columns from the index
# for that one query, which is the situation of the persons recorded in one
# script alone. Every set draws from its own generator seeded with SEED,
# so adding or removing a person changes only that person's queries.
# ─────────────────────────────────────────────────────────────────────────────
if "persons" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

VOWELS = set("aeiouy")
NEIGHBOURS = {   # each letter's neighbouring keys on a QWERTY keyboard
    "q": "wa", "w": "qeas", "e": "wrsd", "r": "etdf", "t": "ryfg", "y": "tugh", "u": "yihj",
    "i": "uojk", "o": "ipkl", "p": "ol", "a": "qwsz", "s": "awedxz", "d": "serfcx",
    "f": "drtgvc", "g": "ftyhbv", "h": "gyujnb", "j": "huikmn", "k": "jiolm", "l": "kop",
    "z": "asx", "x": "zsdc", "c": "xdfv", "v": "cfgb", "b": "vghn", "n": "bhjm", "m": "njk"}

# Appendix C: (rule name, patterns, replacements). Each pattern's first
# group is the replaced part; one match is rewritten to another replacement.
DIASPORA_RULES = [
    ("sch/sz/sh/ch", [r"^(sch|sz|sh|ch)", r"(sch|sz|sh|ch)(?=[bcdfghjklmnpqrstvwxz])"],
     ["sch", "sz", "sh", "ch"]),
    ("j→ch/kh/h", [r"(j)"], ["ch", "kh", "h"]),
    ("v↔w", [r"(v|w)"], ["v", "w"]),
    ("y/i/j initial", [r"^(y|i|j)(?=[aeiou])"], ["y", "i", "j"]),
    ("tz/ts/z/c", [r"(tz|ts|z|c)(?=[ei])"], ["tz", "ts", "z", "c"]),
    ("tz/ts/z", [r"(tz|ts|z)(?![ei])"], ["tz", "ts", "z"]),
    ("k↔c", [r"(k|c)(?=[aou])"], ["k", "c"]),
    ("k↔qu", [r"(k|qu)(?=[ei])"], ["k", "qu"]),
    ("-sky/-ski/-skiy", [r"(skiy|sky|ski)$"], ["sky", "ski", "skiy"]),
    ("-man/-mann", [r"(mann|man)$"], ["man", "mann"]),
    ("-berg/-burg", [r"(berg|burg)$"], ["berg", "burg"]),
    ("-stein/-shtein", [r"(shtein|shtain|stein|stain)$"], ["stein", "shtein", "stain", "shtain"]),
    ("ei/ey/ai", [r"(ei|ey|ai)"], ["ei", "ey", "ai"]),
    ("u↔oo", [r"(oo|u)"], ["u", "oo"]),
    ("f↔ph", [r"(ph|f)"], ["f", "ph"]),
]


def typing_error(word: str, rng: random.Random) -> str:
    """Apply one random typing error (deletion, swap or neighbour key) to `word`."""
    position = rng.randrange(len(word))
    kind = rng.choice(("delete", "swap", "substitute"))
    if kind == "delete" or (kind == "swap" and len(word) < 2):
        return word[:position] + word[position + 1:]
    if kind == "swap":
        position = min(position, len(word) - 2)
        return word[:position] + word[position + 1] + word[position] + word[position + 2:]
    base = unicodedata.normalize("NFKD", word[position])[0].lower()
    replacement = rng.choice(NEIGHBOURS.get(base, "aeiou"))
    replacement = replacement.upper() if word[position].isupper() else replacement
    return word[:position] + replacement + word[position + 1:]


def match_case(template: str, word: str) -> str:
    """Return `word` with the capitalisation pattern of `template`."""
    if template.isupper():
        return word.upper()
    return word.capitalize() if template[:1].isupper() else word.lower()


def rule_variants(word: str, patterns: list[str], replacements: list[str]) -> list[str]:
    """Return every rewrite of `word` by one rule (lower case)."""
    lower, variants = word.lower(), []
    for pattern in patterns:
        for found in re.finditer(pattern, lower):
            variants += [lower[:found.start(1)] + other + lower[found.end(1):]
                         for other in replacements if other != found.group(1)]
    return variants


def doubling_variants(word: str) -> list[str]:
    """Return the rewrites of the last Appendix C rule: a doubled consonant
    made single, or a consonant after a vowel doubled."""
    lower, variants = word.lower(), []
    for i in range(1, len(lower)):
        if lower[i] == lower[i - 1] and lower[i] not in VOWELS:
            variants.append(lower[:i] + lower[i + 1:])
        elif (lower[i] in "bdfgklmnprstz" and lower[i - 1] in VOWELS
              and lower[i + 1:i + 2] != lower[i]):
            variants.append(lower[:i] + lower[i] + lower[i:])
    return variants


def diaspora_variant(word: str, rng: random.Random) -> tuple[str, str] | None:
    """Rewrite `word` with one applicable Appendix C rule chosen at random.

    Returns:
        The variant (in `word`'s capitalisation) and the rule's name, or
        None if no rule applies.
    """
    applicable = [(name, variants) for name, patterns, replacements in DIASPORA_RULES
                  if (variants := rule_variants(word, patterns, replacements))]
    if doubling := doubling_variants(word):
        applicable.append(("doubled consonant", doubling))
    if not applicable:
        return None
    name, variants = rng.choice(applicable)
    return match_case(word, rng.choice(variants)), name


def llm_variant(full_name: str) -> str | None:
    """Ask the LLM for one plausible English, German or Polish spelling of
    `full_name`; only called when ANTHROPIC_API_KEY is set.

    Returns:
        The respelled name, or None if the model declined.
    """
    import anthropic  # pylint: disable=import-outside-toplevel
    client = anthropic.Anthropic()
    response = client.beta.messages.create(
        model=QS2_LLM_MODEL, max_tokens=2000, output_config={"effort": "low"},
        betas=["server-side-fallback-2026-07-01"], fallbacks="default",
        messages=[{"role": "user", "content": (
            "This is the name of a Jewish person buried in Argentina, as spelled in Spanish "
            f"records: {full_name}. Give one plausible spelling of the same name as a "
            "descendant in an English-, German- or Polish-speaking country might type it. "
            "Answer with the respelled name only.")}])
    if response.stop_reason == "refusal":
        return None
    return "".join(block.text for block in response.content if block.type == "text").strip()


def query_token(display: str, source: NameToken) -> dict:
    """Return the JSON record of one query token standing for `source`."""
    return {"display": display, "norm": normalise(display), "script": token_script(display),
            "field": source.field, "source_norm": source.norm}


def make_query(query_set: str, person: Person, tokens: list[dict], **extra) -> dict:
    """Return the CSV row of one query whose only relevant person is its target."""
    return {"query_id": f"{query_set}-{person.person_id}", "set": query_set,
            "query_text": " ".join(t["display"] for t in tokens),
            "query_script": "/".join(sorted({t["script"] for t in tokens})),
            "target_person_ids": json.dumps([person.person_id]),
            "relevant_person_ids": json.dumps([person.person_id]),
            "masked_fields": extra.get("masked_fields", ""),
            "source_fields": extra.get("source_fields", ""),
            "split": person.split, "fold": person.fold or "", "section": person.section,
            "rule": extra.get("rule", ""), "generator": extra.get("generator", ""),
            "parent_query_id": "", "query_tokens": json.dumps(tokens, ensure_ascii=False)}


def name_in_script(person: Person, script: str) -> list[NameToken]:
    """Return the person's given-name then surname tokens in `script`."""
    return ([t for t in person.tokens if t.field == "given" and t.script == script]
            + [t for t in person.tokens if t.field == "surname" and t.script == script])


def has_full_name(person: Person, script: str) -> bool:
    """Return True if `person` has a given name and a surname in `script`."""
    return all(any(t.field == name_field and t.script == script for t in person.tokens)
               for name_field in ("given", "surname"))


def qs1_query(person: Person, rng: random.Random) -> dict:
    """Return the QS1 query of `person`, or an empty dict if no token can take an edit."""
    source = name_in_script(person, "latin")
    if rng.random() < 0.5:
        source = ([t for t in source if t.field == "surname"]
                  + [t for t in source if t.field == "given"])
    editable = [i for i, t in enumerate(source) if len(t.display) >= 3]
    if not editable:
        return {}
    chosen = rng.choice(editable)
    words = [t.display for t in source]
    for _ in range(1 if len(words[chosen]) <= 5 else 2):
        words[chosen] = typing_error(words[chosen], rng)
    return make_query("QS1", person, [query_token(w, t) for w, t in zip(words, source)],
                      source_fields="given_name_es;surname_es", generator="typing errors")


def qs2_query(person: Person, rng: random.Random) -> dict:
    """Return the QS2 query of `person`, or an empty dict if it must be dropped."""
    source = name_in_script(person, "latin")
    words, rules = [], []
    for token in source:
        variant = diaspora_variant(token.display, rng)
        words.append(variant[0] if variant else token.display)
        if variant:
            rules.append(f"{token.field}:{variant[1]}")
    generator = "rules"
    if not rules:
        if preflight["qs2_llm"]["status"] != "ok":
            return {}
        respelled = llm_variant(" ".join(words))
        if not respelled or len(respelled.split()) != len(source):
            return {}
        words, generator = respelled.split(), f"llm:{QS2_LLM_MODEL}"
    return make_query("QS2", person, [query_token(w, t) for w, t in zip(words, source)],
                      source_fields="given_name_es;surname_es", rule="; ".join(rules),
                      generator=generator)


def cross_script_query(query_set: str, person: Person) -> dict:
    """Return the QS3 (Hebrew query) or QS4 (Latin query) query of `person`."""
    script = "hebrew" if query_set == "QS3" else "latin"
    columns = "given_name_he;surname_he" if script == "hebrew" else "given_name_es;surname_es"
    tokens = [query_token(t.display, t) for t in name_in_script(person, script)]
    return make_query(query_set, person, tokens, masked_fields=columns, source_fields=columns,
                      generator=f"recorded {script} name")


def generate(query_set: str, make) -> list[dict]:
    """Return the queries `make` builds for every person with a full Latin name,
    counting in `query_drops` the persons it drops (returns an empty dict for)."""
    rng = random.Random(SEED)
    made = [make(person, rng) for person in persons if has_full_name(person, "latin")]
    query_drops[query_set] = sum(not query for query in made)
    return [query for query in made if query]


query_drops = Counter()
queries = generate("QS1", qs1_query) + generate("QS2", qs2_query)
for _query_set in ("QS3", "QS4"):
    queries += [cross_script_query(_query_set, p) for p in persons
                if has_full_name(p, "latin") and has_full_name(p, "hebrew")]

print(f"{RULE}\nQuery sets QS1–QS4\n{RULE}")
for _query_set in ("QS1", "QS2", "QS3", "QS4"):
    _examples = [q["query_text"] for q in queries if q["set"] == _query_set][:3]
    print(f"  {_query_set}  {sum(q['set'] == _query_set for q in queries):>4} queries  "
          f"{query_drops[_query_set]:>3} dropped   e.g. {' | '.join(_examples)}")
_rules = Counter(rule.split(":", 1)[1] for q in queries if q["set"] == "QS2"
                 for rule in q["rule"].split("; ") if rule)
print(f"{RULE}\n  {len(queries)} queries; QS2 rules used: "
      + ", ".join(f"{name} {count}" for name, count in _rules.most_common()))

────────────────────────────────────────────────────────────────────────
Query sets QS1–QS4
────────────────────────────────────────────────────────────────────────
  QS1   565 queries    0 dropped   e.g. Jacobo Pnius | Arcusin Rqaeul | Vaentn Hurvitz
  QS2   565 queries    0 dropped   e.g. Jakobo Pinoos | Rakel Arcusinn | Valenntin Hoorvitz
  QS3   364 queries    0 dropped   e.g. רחל ארקושין | אב הורוויץ | יוסף קאפמאן
  QS4   364 queries    0 dropped   e.g. Raquel Arcusin | Valentin Hurvitz | Jose Kofman
────────────────────────────────────────────────────────────────────────
  1858 queries; QS2 rules used: doubled consonant 718, u↔oo 76, f↔ph 58, v↔w 52, k↔c 48, j→ch/kh/h 37, -man/-mann 32, -sky/-ski/-skiy 29, ei/ey/ai 25, k↔qu 21, tz/ts/z/c 20, tz/ts/z 20, sch/sz/sh/ch 19, y/i/j initial 16, -berg/-burg 10, -stein/-shtein 3


**3. Query Set QS5 and the Query Files**

This cell takes `queries` and draws the **partial-name** set QS5: `QS5_SIZE` queries sampled from QS1–QS4 with `SEED`, each reduced to a single token, the surname with probability `QS5_SURNAME_P` and otherwise the given name. The surname token is the last one, which for a woman recorded as "X de Y" is her husband's surname Y. Since a single common name matches many people, a QS5 query's relevant persons are every person whose normalised token in the same field and script equals the recorded token, plus the target; the parent query's masking still applies. Relevance for a development query looks only at development persons. A test query is evaluated against an index of every person, so its relevant persons are drawn from all of them. It writes:

1. `./eval/queries_dev.csv` and `./eval/queries_test.csv`: every query of its split, QS1–QS5
2. `./review/qs2_sample.csv`: `QS2_REVIEW_SIZE` random QS2 variants beside the recorded names, with an empty `plausible` column for the reviewer

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Query set QS5 and the query files  —  single-token queries, many relevant
#
# The QS5 query text is the parent's token, so a typing error or a variant
# in the kept token stays in it; relevance is decided on the recorded token
# it stands for (`source_norm`), in the script that token was recorded in.
# ─────────────────────────────────────────────────────────────────────────────
if "queries" not in globals():
    raise RuntimeError("Run the 'Query Sets QS1–QS4' cell first.")


def token_owners(candidates: list[Person]) -> dict[tuple, set[str]]:
    """Map each (field, script, norm) to the persons holding such a token."""
    owners = defaultdict(set)
    for person in candidates:
        for token in person.tokens:
            owners[(token.field, token.script, token.norm)].add(person.person_id)
    return owners


def qs5_query(parent: dict, rng: random.Random, owners: dict) -> dict:
    """Reduce `parent` to one token and list every person relevant to it."""
    field_name = "surname" if rng.random() < QS5_SURNAME_P else "given"
    tokens = [t for t in json.loads(parent["query_tokens"]) if t["field"] == field_name]
    kept = tokens[-1] if field_name == "surname" else tokens[0]
    (target,) = json.loads(parent["target_person_ids"])
    source_script = next(t.script for t in persons_by_id[target].tokens
                         if t.field == field_name and t.norm == kept["source_norm"])
    relevant = owners[(field_name, source_script, kept["source_norm"])] | {target}
    return {**parent, "query_id": f"QS5-{parent['set']}-{target}", "set": "QS5",
            "query_text": kept["display"], "query_script": kept["script"],
            "relevant_person_ids": json.dumps(sorted(relevant)),
            "parent_query_id": parent["query_id"],
            "query_tokens": json.dumps([kept], ensure_ascii=False)}


_rng = random.Random(SEED)
_owners = {"dev": token_owners(dev_persons), "test": token_owners(persons)}
_parents = list(queries)
_rng.shuffle(_parents)
queries += [qs5_query(parent, _rng, _owners[parent["split"]]) for parent in _parents[:QS5_SIZE]]

for _split in ("dev", "test"):
    write_csv(EVAL_DIR / f"queries_{_split}.csv", [q for q in queries if q["split"] == _split])
_qs2 = [q for q in queries if q["set"] == "QS2"]
random.Random(SEED).shuffle(_qs2)
write_csv(REVIEW_DIR / "qs2_sample.csv", [
    {"query_id": q["query_id"],
     "recorded_name": " ".join(t.display for t in name_in_script(
         persons_by_id[json.loads(q["target_person_ids"])[0]], "latin")),
     "variant": q["query_text"], "rule": q["rule"], "generator": q["generator"],
     "plausible": ""}
    for q in _qs2[:QS2_REVIEW_SIZE]])
query_counts = Counter((q["set"], q["split"]) for q in queries)

print(f"{RULE}\nQuery files: {EVAL_DIR}/queries_<split>.csv\n{RULE}")
for _query_set in ("QS1", "QS2", "QS3", "QS4", "QS5"):
    print(f"  {_query_set}  dev {query_counts[(_query_set, 'dev')]:>4}   "
          f"test {query_counts[(_query_set, 'test')]:>4}")
_qs5 = [q for q in queries if q["set"] == "QS5"]
print(f"{RULE}\n  {len(queries)} queries; QS5 has "
      f"{np.mean([len(json.loads(q['relevant_person_ids'])) for q in _qs5]):.1f} relevant "
      f"persons per query on average")

────────────────────────────────────────────────────────────────────────
Query files: eval/queries_<split>.csv
────────────────────────────────────────────────────────────────────────
  QS1  dev  450   test  115
  QS2  dev  450   test  115
  QS3  dev  300   test   64
  QS4  dev  300   test   64
  QS5  dev  243   test   57
────────────────────────────────────────────────────────────────────────
  2158 queries; QS5 has 6.2 relevant persons per query on average


**4. Consistency Check**

This cell reads the two query files back from disk and checks them against the persons files, independently of the cells above:

1. the normalisation copied into this notebook reproduces the stored normalised form of every person's token, so that query tokens and index tokens are normalised alike
2. every query's target is a person of its split, and every masked column belongs to the target
3. a QS1–QS4 query's only relevant person is its target, and every QS5 relevant list equals the one recomputed from the persons files
4. the development file names no test person, as target or as relevant person, which seals the test split

It ends with the queries per split and the result of the check.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the query files against the persons files
#
# The persons are re-read from disk as plain dictionaries, and the QS5
# relevant lists are recomputed from them with a separate token index.
# ─────────────────────────────────────────────────────────────────────────────
if "query_counts" not in globals():
    raise RuntimeError("Run every cell above first.")


def normaliser_problems(stored: dict[str, dict]) -> list[str]:
    """Check that `normalise` reproduces every stored token's normalised form."""
    return [f"{pid}: {t['display']} → {normalise(t['display'])}, stored {t['norm']}"
            for pid, row in stored.items() for t in json.loads(row["tokens"])
            if normalise(t["display"]) != t["norm"]]


def query_problems(stored: dict[str, dict], split: str) -> list[str]:
    """Check one split's query file against the stored persons."""
    in_split = {pid for pid, row in stored.items() if split == "test" or row["split"] == "dev"}
    owners = defaultdict(set)
    for person_id in in_split:
        for token in json.loads(stored[person_id]["tokens"]):
            owners[(token["field"], token["script"], token["norm"])].add(person_id)
    problems = []
    for query in read_csv(EVAL_DIR / f"queries_{split}.csv"):
        (target,) = json.loads(query["target_person_ids"])
        relevant = set(json.loads(query["relevant_person_ids"]))
        if target not in stored or stored[target]["split"] != split:
            problems.append(f"{query['query_id']}: target {target} is not a {split} person")
            continue
        target_tokens = json.loads(stored[target]["tokens"])
        if not set(filter(None, query["masked_fields"].split(";"))) <= {
                t["column"] for t in target_tokens}:
            problems.append(f"{query['query_id']}: masks columns its target lacks")
        expected = {target}
        if query["set"] == "QS5":
            (kept,) = json.loads(query["query_tokens"])
            script = next(t["script"] for t in target_tokens
                          if t["field"] == kept["field"] and t["norm"] == kept["source_norm"])
            expected |= owners[(kept["field"], script, kept["source_norm"])]
        if relevant != expected:
            problems.append(f"{query['query_id']}: relevant list differs when recomputed")
        if split == "dev" and any(stored.get(p, {}).get("split") != "dev" for p in relevant):
            problems.append(f"{query['query_id']}: a test person in a development file")
    return problems


_stored = {row["person_id"]: row for split in ("dev", "test")
           for row in read_csv(EVAL_DIR / f"persons_{split}.csv")}
_checks = {"normalisation copy": normaliser_problems(_stored),
           "queries_dev.csv": query_problems(_stored, "dev"),
           "queries_test.csv": query_problems(_stored, "test")}
print(f"{RULE}\nConsistency: {EVAL_DIR}/queries_<split>.csv against the persons files\n{RULE}")
for _name, _problems in _checks.items():
    print(f"  {'!! ' if _problems else 'OK '}{_name:<20} "
          + (f"{len(_problems)} problem(s)" if _problems else "consistent"))
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
_consistent = not any(_checks.values())
print(f"{RULE}\n  {sum(q['split'] == 'dev' for q in queries)} development and "
      f"{sum(q['split'] == 'test' for q in queries)} test queries  "
      f"({'consistent' if _consistent else 'inconsistent'})")
print("\n✓ Query sets complete (QS1–QS5 for both splits, QS2 sample for review).")
print(f"  query files consistent: {_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: eval/queries_<split>.csv against the persons files
────────────────────────────────────────────────────────────────────────
  OK normalisation copy   consistent
  OK queries_dev.csv      consistent
  OK queries_test.csv     consistent
────────────────────────────────────────────────────────────────────────
  1743 development and 415 test queries  (consistent)

✓ Query sets complete (QS1–QS5 for both splits, QS2 sample for review).
  query files consistent: True
